In [1]:
import importlib
import sys
import torch
import numpy as np

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')

In [2]:
import event_log_loader.new_event_log_loader_v2
importlib.reload(event_log_loader.new_event_log_loader_v2)
from event_log_loader.new_event_log_loader_v2 import PrefixesDataFrameLoader, EventLogLoader

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

np.random.seed(17)

event_log_location = '../../../../../../data/data/helpdesk.csv'

result_name = 'helpdesk_all'

cat_dynamic = ['Activity', 'Resource']
# cat_static =  ['VariantIndex', 'seriousness', 'customer', 'product', 'responsible_section', 'seriousness_2', 'service_level', 'service_type', 'support_section', 'workgroup']
cat_static = ['customer', 'product', 'seriousness']

num_dynamic = ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day']
num_dyn_log = []
num_static = []

event_log_properties = {# case id
                        'case_name' : 'CaseID',
                        # activit
                        'concept_name' : 'Activity',
                        # time values and computaitons
                        'timestamp_name' : 'CompleteTimestamp',
                        'date_format' : '%Y/%m/%d %H:%M:%S.%f',
                        'time_since_case_start_column' : 'case_elapsed_time',
                        'time_since_last_event_column' : 'event_elapsed_time',
                        'day_in_week_column' : 'day_in_week',
                        'seconds_in_day_column' : 'seconds_in_day',
                        # min suffix size for eos padding right
                        'min_suffix_size' : 5,
                        # trian and test split
                        'train_validation_size' : 0.15,
                        'test_validation_size' : 0.2,
                        # window size for padding
                        'window_size' : 'auto',
                        # dynamic and static values
                        'categorical_columns' : cat_dynamic,
                        'static_categorical_columns' : cat_static,
                        'continuous_columns' : num_dynamic,
                        'continuous_positive_columns' : num_dyn_log,
                        'static_continuous_columns' : num_static
                        }


In [3]:
# object to create datframe of prefixes for petri-net repaly marking computation
pref_adopt_dataframe = PrefixesDataFrameLoader(event_log_location=event_log_location, event_log_properties=event_log_properties)

In [4]:
train_pref_df = pref_adopt_dataframe.get_dataset('train')
train_pref_df

,CaseID,prefix_length,Activity,Resource,case_elapsed_time,event_elapsed_time,day_in_week,seconds_in_day,customer,product,seriousness
0,Case 10,1,[Assign seriousness],[Value 2],[0.0],[nan],[2.0],[31820.0],Value 10,Value 3,Value 1
1,Case 10,2,"[Assign seriousness, Take in charge ticket]","[Value 2, Value 2]","[0.0, 3196606.0]","[nan, 3196606.0]","[2.0, 4.0]","[31820.0, 31626.0]",Value 10,Value 3,Value 1
2,Case 10,3,"[Assign seriousness, Take in charge ticket, Re...","[Value 2, Value 2, Value 2]","[0.0, 3196606.0, 3196613.0]","[nan, 3196606.0, 7.0]","[2.0, 4.0, 4.0]","[31820.0, 31626.0, 31633.0]",Value 10,Value 3,Value 1
3,Case 100,1,[Assign seriousness],[Value 1],[0.0],[nan],[4.0],[37517.0],Value 44,Value 1,Value 1
4,Case 100,2,"[Assign seriousness, Take in charge ticket]","[Value 1, Value 9]","[0.0, 1036724.0]","[nan, 1036724.0]","[4.0, 2.0]","[37517.0, 37441.0]",Value 44,Value 1,Value 1
...,...,...,...,...,...,...,...,...,...,...,...
10936,Case 993,4,"[Assign seriousness, Take in charge ticket, Wa...","[Value 8, Value 8, Value 2, Value 2]","[0.0, 235.0, 1200632.0, 1379474.0]","[nan, 235.0, 1200397.0, 178842.0]","[0.0, 0.0, 0.0, 2.0]","[40283.0, 40518.0, 31315.0, 37357.0]",Value 213,Value 2,Value 1
10937,Case 993,5,"[Assign seriousness, Take in charge ticket, Wa...","[Value 8, Value 8, Value 2, Value 2, Value 2]","[0.0, 235.0, 1200632.0, 1379474.0, 1379486.0]","[nan, 235.0, 1200397.0, 178842.0, 12.0]","[0.0, 0.0, 0.0, 2.0, 2.0]","[40283.0, 40518.0, 31315.0, 37357.0, 37369.0]",Value 213,Value 2,Value 1
10938,Case 996,1,[Assign seriousness],[Value 8],[0.0],[nan],[3.0],[49903.0],Value 9,Value 3,Value 1
10939,Case 996,2,"[Assign seriousness, Take in charge ticket]","[Value 8, Value 6]","[0.0, 325091.0]","[nan, 325091.0]","[3.0, 0.0]","[49903.0, 29394.0]",Value 9,Value 3,Value 1


In [5]:
event_log_loader = EventLogLoader(event_log_location=event_log_location, event_log_properties=event_log_properties, prefix_df=pref_adopt_dataframe)

In [6]:
train_dataset = event_log_loader.get_dataset('train')
torch.save(train_dataset, '../../../../encoded_data/'+result_name+'_'+str(event_log_loader.encoder_decoder.min_suffix_size)+'_train.pkl')

categorical tensors:   0%|          | 0/2 [00:00<?, ?it/s]

Activity:   0%|          | 0/2977 [00:00<?, ?it/s]

Resource:   0%|          | 0/2977 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/4 [00:00<?, ?it/s]

case_elapsed_time:   0%|          | 0/2977 [00:00<?, ?it/s]

event_elapsed_time:   0%|          | 0/2977 [00:00<?, ?it/s]

day_in_week:   0%|          | 0/2977 [00:00<?, ?it/s]

seconds_in_day:   0%|          | 0/2977 [00:00<?, ?it/s]

In [7]:
train_dataset

In [8]:
print(train_dataset.all_categories)
print(train_dataset.all_static_categories)

print(train_dataset.categorical_tensors[0].size())
print(train_dataset.continuous_tensors[0].size())

print(train_dataset.static_categorical_tensor.size())
print(train_dataset.static_continuous_tensor.size())

print(train_dataset.zero_padding.size())
print(train_dataset.eos_padding.size())

print(train_dataset.zero_padding[0:5])
print(train_dataset.eos_padding[0:5])

([('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge ticket': 13, 'VERIFIED': 14, 'Wait': 15}), ('Resource', 24, {'EOS': 1, 'Value 1': 2, 'Value 10': 3, 'Value 11': 4, 'Value 12': 5, 'Value 13': 6, 'Value 14': 7, 'Value 15': 8, 'Value 16': 9, 'Value 17': 10, 'Value 18': 11, 'Value 19': 12, 'Value 2': 13, 'Value 20': 14, 'Value 21': 15, 'Value 22': 16, 'Value 3': 17, 'Value 4': 18, 'Value 5': 19, 'Value 6': 20, 'Value 7': 21, 'Value 8': 22, 'Value 9': 23})], [('case_elapsed_time', 1, {}), ('event_elapsed_time', 1, {}), ('day_in_week', 1, {}), ('seconds_in_day', 1, {})])
([('customer', 360, {'Value 1': 1, 'Value 10': 2, 'Value 100': 3, 'Value 101': 4, 'Value 102': 5, 'Value 103': 6, 'Value 104': 7, 'Value 105': 8, 'Value 106': 9, 'Value 107': 10, 'Value 108': 11, 'Value 11